# MOCS Excavator Detection — YOLO11n
Run this notebook with a Kaggle GPU accelerator. Attach both the MOCS dataset and a Kaggle dataset containing this repository, or clone the repository into `/kaggle/working`. The workflow deliberately inspects and visualizes annotations before training.

In [ ]:
!nvidia-smi
!python -m pip install -q 'ultralytics>=8.3.0,<9' 'streamlit>=1.39,<2' 'Pillow>=10.4,<12' 'PyYAML>=6,<7'

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

working_project = Path('/kaggle/working/excavator-detection')
candidates = [Path.cwd(), working_project] + [p.parent.parent for p in Path('/kaggle/input').glob('**/scripts/prepare_dataset.py')]
source_project = next((p for p in candidates if (p / 'scripts/prepare_dataset.py').is_file()), None)
if source_project is None:
    raise FileNotFoundError('Attach this repository as a Kaggle input or clone it under /kaggle/working.')
if source_project.resolve() != working_project.resolve():
    shutil.copytree(source_project, working_project, dirs_exist_ok=True)
PROJECT = working_project
print('Project:', PROJECT)
print('Python:', sys.version)

In [ ]:
# Locate the attached Kaggle dataset. Set this explicitly if auto-detection is ambiguous.
dataset_candidates = [p for p in Path('/kaggle/input').iterdir() if 'mocs' in p.name.lower()]
if len(dataset_candidates) != 1:
    raise RuntimeError(f'Expected one MOCS input; found: {dataset_candidates}. Set RAW_DATA manually.')
RAW_DATA = dataset_candidates[0]
print('Raw data:', RAW_DATA)

## 1. Inspect and visualize
Read the JSON report carefully. Confirm that an Excavator class is listed and inspect the saved sample figures before continuing.

In [ ]:
subprocess.run([sys.executable, 'scripts/inspect_dataset.py', '--data', str(RAW_DATA), '--samples', '12'], cwd=PROJECT, check=True)

In [ ]:
from IPython.display import display
from PIL import Image
for sample in sorted((PROJECT / 'figures/dataset_samples').glob('*.jpg'))[:6]:
    image = Image.open(sample); image.thumbnail((700, 700)); display(image)

## 2. Convert to one-class YOLO
This preserves a complete official split if found; otherwise it uses a deterministic 70/15/15 split with seed 42.

In [ ]:
subprocess.run([sys.executable, 'scripts/prepare_dataset.py', '--data', str(RAW_DATA), '--seed', '42', '--image-mode', 'symlink'], cwd=PROJECT, check=True)

## 3. Required one-epoch smoke test

In [ ]:
subprocess.run([sys.executable, 'scripts/train.py', '--smoke-test', '--device', '0', '--workers', '2'], cwd=PROJECT, check=True)

## 4. Full 50-epoch run
Only run after the smoke test completes and its labels look correct. Early stopping uses patience 10.

In [ ]:
subprocess.run([sys.executable, 'scripts/train.py', '--epochs', '50', '--imgsz', '640', '--device', '0', '--workers', '2', '--seed', '42'], cwd=PROJECT, check=True)

## 5. Held-out test evaluation

In [ ]:
best = PROJECT / 'results/train/yolo11n_excavator/weights/best.pt'
subprocess.run([sys.executable, 'scripts/evaluate.py', '--weights', str(best), '--device', '0'], cwd=PROJECT, check=True)
print((PROJECT / 'results/evaluation/test/metrics.json').read_text())

## 6. Example predictions and artifact export

In [ ]:
test_images = PROJECT / 'data/processed/images/test'
subprocess.run([sys.executable, 'scripts/predict.py', '--source', str(test_images), '--weights', str(best), '--name', 'test_examples'], cwd=PROJECT, check=True)
shutil.make_archive('/kaggle/working/excavator_results', 'zip', PROJECT / 'results')
print('Download /kaggle/working/excavator_results.zip and the best.pt checkpoint from the Files panel.')